In [11]:
import pandas as pd
import numpy as np
from joblib import load
from preprocessor import *
from Preprocessor_LLM import *
from Preprocessor_LLM_RAG import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from transformers import pipeline
#import torch
import re
import sys
import os
from openai import OpenAI


In [12]:
# Installation des librairies de base
!pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 17.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 25.0 MB/s  0:00:18m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 54.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 53.3 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 52.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 28.1 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 44.4 MB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 68.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 45.4 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 70.0 MB/s

In [14]:
# PROMPT : COSTAR METHOD


PROMPT_CONFIG = {
    # ========== STRATÉGIE GLOBALE ==========
    "objectif": "Sélectionner les cas les plus utiles pour améliorer la détection de fraude",
    
    # ========== CE QUE LE LLM DOIT PRIVILÉGIER ==========
    "priorites": [
        "Chercher des profils très différents les uns des autres",
        "Inclure des cas suspects avec des combinaisons inhabituelles",
        "Couvrir tout le spectre de probabilités (pas seulement le milieu)",
        "Ajouter quelques cas extrêmes et atypiques"
    ],
    
    # ========== CE QUE LE LLM DOIT ÉVITER ==========
    "a_eviter": [
        "Sélectionner 50 profils similaires",
        "Se concentrer uniquement sur les cas moyens",
        "Ignorer les cas rares ou extrêmes"
    ],
    
    # ========== EXEMPLES DE PATTERNS À DÉTECTER ==========
    "patterns_interessants": [
        "Jeune conducteur + véhicule cher",
        "Pas de témoin + pas de rapport police",
        "Délai très court ou très long avant déclaration",
        "Changement d'adresse récent"
    ],
    
    # ========== PARAMÈTRES NUMÉRIQUES ==========
    "n_preselect": 500,  # Combien présélectionner par uncertainty sampling
    "n_select": 50,      # Combien sélectionner au final
}


def generate_costar_prompt(config):
    prompt = f"""
    CONTEXT: Je travaille sur la détection de fraude.
    OBJECTIVE: {config['objectif']} (Passer de {config['n_preselect']} à {config['n_select']} cas).
    STYLE: Analyste expert en fraude.
    PRIORITIES: {', '.join(config['priorites'])}.
    PATTERNS TO DETECT: {', '.join(config['patterns_interessants'])}.
    AVOID: {', '.join(config['a_eviter'])}.
    RESPONSE: Liste des {config['n_select']} IDs avec justification.
    """
    return prompt

# Utilisation
final_prompt = generate_costar_prompt(PROMPT_CONFIG)
final_prompt

"\n    CONTEXT: Je travaille sur la détection de fraude.\n    OBJECTIVE: Sélectionner les cas les plus utiles pour améliorer la détection de fraude (Passer de 500 à 50 cas).\n    STYLE: Analyste expert en fraude.\n    PRIORITIES: Chercher des profils très différents les uns des autres, Inclure des cas suspects avec des combinaisons inhabituelles, Couvrir tout le spectre de probabilités (pas seulement le milieu), Ajouter quelques cas extrêmes et atypiques.\n    PATTERNS TO DETECT: Jeune conducteur + véhicule cher, Pas de témoin + pas de rapport police, Délai très court ou très long avant déclaration, Changement d'adresse récent.\n    AVOID: Sélectionner 50 profils similaires, Se concentrer uniquement sur les cas moyens, Ignorer les cas rares ou extrêmes.\n    RESPONSE: Liste des 50 IDs avec justification.\n    "

In [15]:
data = pd.read_csv('/home/onyxia/PROJET_STATAPP/Data/Cleans/Data_for_active_learning.csv')

X = data.drop(columns=['FraudFound_P'])
y = data['FraudFound_P']


In [16]:
preprocessor = load("preprocessor.joblib")
print("✓ Preprocessor numérique chargé")

preprocessor_llm = PreprocessLLM(
    label_cols=label_cols,
    freq_cols=freq_cols,
    ordinal_cols=ordinal_cols,
    binary_cols=binary_cols,
    scale_cols=scale_cols
)

all_cols = binary_cols + label_cols + freq_cols + ordinal_cols + scale_cols
X_preprocessed = preprocessor.fit_transform(X)
X_preprocessed = pd.DataFrame(X_preprocessed, columns=all_cols)

X_raw = X.copy()

print(f"  Données numériques: {X_preprocessed.shape}")
print(f"  Données brutes (pour LLM): {X_raw.shape}")

exemple_text = preprocessor_llm.transform(X_raw.head(1))
print("\n Exemple preprocessing LLM:")
print(exemple_text[0]['text'])

✓ Preprocessor numérique chargé
  Données numériques: (15419, 20)
  Données brutes (pour LLM): (15419, 21)

 Exemple preprocessing LLM:
Âge : 21 ans
Délai avant déclaration : 1 semaines
Sexe : Female
Zone de l'accident : Urban
Responsabilité : Policy Holder
Rapport de police : No
Présence de témoins : No
Type d'agent : External
VehiclePrice : more than 69000
Days_Policy_Accident : more than 30
PastNumberOfClaims : none
AgeOfVehicle : 3 years
NumberOfSuppliments : none
AddressChange_Claim : 1 year
NumberOfCars : 3 to 4
DriverRating : 1
Deductible : 300
PolicyType : Sport - Liability
Make : Honda
MaritalStatus : Single


In [17]:
# Preprocessor for RAG

Fraud_col = "FraudFound_P"

preprocessor_llm_rag = PreprocessLLM_RAG(
    label_cols=label_cols,
    freq_cols=freq_cols,
    ordinal_cols=ordinal_cols,
    binary_cols=binary_cols,
    scale_cols=scale_cols,
    Fraud_col = Fraud_col
)

exemple_text = preprocessor_llm_rag.transform_rag(data.head(1))
print("\n Exemple preprocessing LLM RAG:")
print(exemple_text[0]['text'])


 Exemple preprocessing LLM RAG:
Âge : 21 ans
Délai avant déclaration : 1 semaines
Sexe : Female
Zone de l'accident : Urban
Responsabilité : Policy Holder
Rapport de police : No
Présence de témoins : No
Type d'agent : External
VehiclePrice : more than 69000
Days_Policy_Accident : more than 30
PastNumberOfClaims : none
AgeOfVehicle : 3 years
NumberOfSuppliments : none
AddressChange_Claim : 1 year
NumberOfCars : 3 to 4
DriverRating : 1
Deductible : 300
PolicyType : Sport - Liability
Make : Honda
MaritalStatus : Single
etat de Fraude : 0


In [18]:
# Split des données

# 20% pour le test
X_pool_full, X_test, y_pool_full, y_test = train_test_split(
    X_preprocessed, y, test_size=0.2, stratify=y, random_state=42
)

# De même pour les raw
X_raw_pool_full, X_raw_test, _, _ = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

# 2% d'observations initiales pour le seed
X_init, X_pool, y_init, y_pool = train_test_split(
    X_pool_full, y_pool_full, train_size=0.02, stratify=y_pool_full, random_state=42
)

# De même pour les raw
X_raw_init, X_raw_pool, _, _ = train_test_split(
    X_raw_pool_full, y_pool_full, train_size=0.02, stratify=y_pool_full, random_state=42
)

# Données pour uncertainty sampling
X_init_US = X_init.copy()
X_pool_US = X_pool.copy()
y_init_US = y_init.copy()
y_pool_US = y_pool.copy()

# Données pour LLM sampling
X_init_LLM = X_init.copy()
X_pool_LLM = X_pool.copy()
X_raw_pool_LLM = X_raw_pool.copy()
y_init_LLM = y_init.copy()
y_pool_LLM = y_pool.copy()

print(f"  Test set: {len(X_test)} observations")
print(f"  Pool initial: {len(X_pool)} observations")
print(f"  Seed initial: {len(X_init)} observations ({len(X_init)/len(X_pool_full)*100:.1f}%)")

  Test set: 3084 observations
  Pool initial: 12089 observations
  Seed initial: 246 observations (2.0%)


In [19]:
# Initialisation des modèles et du SMOTE (par rapport à l'AL, ajout du model_LLM)

model = LGBMClassifier(
    learning_rate=0.05,
    max_depth=10,
    n_estimators=300,
    random_state=42
)

model_US = LGBMClassifier(
    learning_rate=0.05,
    max_depth=10,
    n_estimators=300,
    random_state=42
)

model_LLM = LGBMClassifier(
    learning_rate=0.05,
    max_depth=10,
    n_estimators=300,
    random_state=42
)

# SMOTE
smote = SMOTE(sampling_strategy=0.25, random_state=42)

In [21]:
# ===========================
# FONCTIONS UTILITAIRES
# ===========================
def enumerate_observations(text_samples):
    summary = ""
    for i, sample in enumerate(text_samples[:len(text_samples)]):
        summary += f"--- observations {i} --- \n{sample['text']}\n\n"

def create_summary_for_llm(text_samples, n_examples=3):
    """Crée un résumé lisible des échantillons pour le LLM"""
    n_examples = len(text_samples)
    summary = f"Vous avez {len(text_samples)} échantillons présélectionnés.\n\n"
    summary += f"Exemples de {min(n_examples, len(text_samples))} premiers échantillons:\n\n"
    for i, sample in enumerate(text_samples[:n_examples]):
        summary += f"--- Échantillon {i} ---\n{sample['text']}\n\n"
    return summary

def create_llm_prompt(text_samples, n_select=50):
    """Crée le prompt pour le LLM"""
    summary = create_summary_for_llm(text_samples)
    
    prompt = f"""Tu es un expert en apprentissage actif pour la détection de fraude à l'assurance.
Tu dois sélectionner les {n_select} échantillons les plus informatifs parmi {len(text_samples)} candidats présélectionnés.

Critères de sélection:
1. Maximiser l'incertitude (cas ambigus, proches de la frontière de décision)
2. Assurer la diversité des profils (âges, types de véhicules, circonstances variées)
3. Prioriser les patterns inhabituels qui pourraient révéler des fraudes

{summary}

Basé sur ces principes, fournis UNIQUEMENT une liste Python des {n_select} indices (de 0 à {len(text_samples)-1}) les plus pertinents à labelliser.

Aussi, sache que certaines colonnes ont été normalisees, c'est qui pourrait expliquer des valeurs qui peuvent te sembler louches.

Enfin, epargne moi des explications en donnant la reponse. Donnes juste ce qui t'ai demande comme reponse.

Format attendu: [0, 5, 12, 18, 23, ...]
"""
    
    return prompt

def parse_llm_response(llm_output, n_max, fallback_n=50):
    """Parse la réponse du LLM pour extraire les indices"""
    try:
        match = re.search(r'\[[\d,\s]+\]', llm_output)
        if match:
            indices = eval(match.group())
            indices = [int(i) for i in indices if isinstance(i, (int, float)) and 0 <= int(i) < n_max]
            indices = indices[:fallback_n]
            
            if len(indices) < fallback_n:
                remaining = [i for i in range(n_max) if i not in indices]
                np.random.shuffle(remaining)
                indices.extend(remaining[:fallback_n - len(indices)])
            
            print(f"  ✓ {len(indices)} indices extraits")
            return indices
        else:
            print("  ⚠ Aucune liste trouvée, fallback")
            return list(range(min(fallback_n, n_max)))
    except Exception as e:
        print(f"  ⚠ Erreur parsing: {e}")
        return list(range(min(fallback_n, n_max)))

print("✓ Fonctions utilitaires définies")

✓ Fonctions utilitaires définies


In [23]:
# Un exemple d'observation
example_dobservation = X_pool_full.iloc[:10]
example_dobservation

,AccidentArea,Sex,Fault,PoliceReportFiled,WitnessPresent,AgentType,Make,MaritalStatus,PolicyType,VehiclePrice,Days_Policy_Accident,PastNumberOfClaims,AgeOfVehicle,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,DriverRating,Deductible,Age,delay_weeks
6619,0.0,1.0,1.0,0.0,0.0,0.0,13.0,2.0,0.264998,1.0,4.0,1.0,4.0,0.0,0.0,0.0,1.0,400.0,-0.884645,-1.815382
9591,1.0,1.0,1.0,0.0,0.0,0.0,17.0,1.0,0.264998,1.0,4.0,2.0,6.0,3.0,0.0,0.0,3.0,400.0,0.221230,-1.200758
2061,1.0,1.0,1.0,0.0,0.0,0.0,9.0,1.0,0.323432,1.0,4.0,0.0,5.0,2.0,0.0,0.0,3.0,400.0,-0.726663,0.561165
14862,1.0,1.0,0.0,0.0,0.0,0.0,13.0,1.0,0.323432,1.0,4.0,2.0,6.0,3.0,0.0,0.0,1.0,400.0,-0.015743,-1.692458
4997,1.0,1.0,1.0,0.0,0.0,0.0,15.0,2.0,0.323432,2.0,4.0,3.0,6.0,3.0,3.0,1.0,2.0,500.0,-0.252716,0.602140
9187,1.0,1.0,1.0,0.0,0.0,0.0,17.0,1.0,0.264998,0.0,4.0,2.0,6.0,3.0,0.0,0.0,4.0,400.0,0.695177,0.602140
1901,1.0,1.0,0.0,0.0,0.0,0.0,9.0,1.0,0.362151,1.0,4.0,1.0,6.0,0.0,0.0,0.0,2.0,400.0,0.616186,0.561165
1894,1.0,0.0,1.0,0.0,0.0,0.0,13.0,2.0,0.323432,2.0,4.0,3.0,5.0,0.0,0.0,0.0,4.0,400.0,-0.805654,0.561165
14656,1.0,1.0,1.0,0.0,0.0,0.0,6.0,2.0,0.362151,5.0,4.0,2.0,0.0,0.0,0.0,0.0,4.0,400.0,-1.911529,0.192390
4317,1.0,1.0,1.0,0.0,0.0,0.0,5.0,1.0,0.323432,2.0,4.0,2.0,7.0,0.0,0.0,0.0,3.0,400.0,1.090132,0.561165


In [24]:
# transformation en dictionnaire 
example_dobservation = preprocessor_llm.transform(example_dobservation.reset_index(drop = True))
example_dobservation

[{'id': 0,
  'text': "Âge : -0.8846449683841019 ans\nDélai avant déclaration : -1.8153824604852427 semaines\nSexe : 1.0\nZone de l'accident : 0.0\nResponsabilité : 1.0\nRapport de police : 0.0\nPrésence de témoins : 0.0\nType d'agent : 0.0\nVehiclePrice : 1.0\nDays_Policy_Accident : 4.0\nPastNumberOfClaims : 1.0\nAgeOfVehicle : 4.0\nNumberOfSuppliments : 0.0\nAddressChange_Claim : 0.0\nNumberOfCars : 0.0\nDriverRating : 1.0\nDeductible : 400.0\nPolicyType : 0.2649977300732862\nMake : 13.0\nMaritalStatus : 2.0"},
 {'id': 1,
  'text': "Âge : 0.2212304022115742 ans\nDélai avant déclaration : -1.2007581958851359 semaines\nSexe : 1.0\nZone de l'accident : 1.0\nResponsabilité : 1.0\nRapport de police : 0.0\nPrésence de témoins : 0.0\nType d'agent : 0.0\nVehiclePrice : 1.0\nDays_Policy_Accident : 4.0\nPastNumberOfClaims : 2.0\nAgeOfVehicle : 6.0\nNumberOfSuppliments : 3.0\nAddressChange_Claim : 0.0\nNumberOfCars : 0.0\nDriverRating : 3.0\nDeductible : 400.0\nPolicyType : 0.2649977300732862\nM

In [25]:
create_summary_for_llm(example_dobservation, 10)

"Vous avez 10 échantillons présélectionnés.\n\nExemples de 10 premiers échantillons:\n\n--- Échantillon 0 ---\nÂge : -0.8846449683841019 ans\nDélai avant déclaration : -1.8153824604852427 semaines\nSexe : 1.0\nZone de l'accident : 0.0\nResponsabilité : 1.0\nRapport de police : 0.0\nPrésence de témoins : 0.0\nType d'agent : 0.0\nVehiclePrice : 1.0\nDays_Policy_Accident : 4.0\nPastNumberOfClaims : 1.0\nAgeOfVehicle : 4.0\nNumberOfSuppliments : 0.0\nAddressChange_Claim : 0.0\nNumberOfCars : 0.0\nDriverRating : 1.0\nDeductible : 400.0\nPolicyType : 0.2649977300732862\nMake : 13.0\nMaritalStatus : 2.0\n\n--- Échantillon 1 ---\nÂge : 0.2212304022115742 ans\nDélai avant déclaration : -1.2007581958851359 semaines\nSexe : 1.0\nZone de l'accident : 1.0\nResponsabilité : 1.0\nRapport de police : 0.0\nPrésence de témoins : 0.0\nType d'agent : 0.0\nVehiclePrice : 1.0\nDays_Policy_Accident : 4.0\nPastNumberOfClaims : 2.0\nAgeOfVehicle : 6.0\nNumberOfSuppliments : 3.0\nAddressChange_Claim : 0.0\nNumb

In [26]:
prompt = create_llm_prompt(example_dobservation, 4)
prompt

"Tu es un expert en apprentissage actif pour la détection de fraude à l'assurance.\nTu dois sélectionner les 4 échantillons les plus informatifs parmi 10 candidats présélectionnés.\n\nCritères de sélection:\n1. Maximiser l'incertitude (cas ambigus, proches de la frontière de décision)\n2. Assurer la diversité des profils (âges, types de véhicules, circonstances variées)\n3. Prioriser les patterns inhabituels qui pourraient révéler des fraudes\n\nVous avez 10 échantillons présélectionnés.\n\nExemples de 10 premiers échantillons:\n\n--- Échantillon 0 ---\nÂge : -0.8846449683841019 ans\nDélai avant déclaration : -1.8153824604852427 semaines\nSexe : 1.0\nZone de l'accident : 0.0\nResponsabilité : 1.0\nRapport de police : 0.0\nPrésence de témoins : 0.0\nType d'agent : 0.0\nVehiclePrice : 1.0\nDays_Policy_Accident : 4.0\nPastNumberOfClaims : 1.0\nAgeOfVehicle : 4.0\nNumberOfSuppliments : 0.0\nAddressChange_Claim : 0.0\nNumberOfCars : 0.0\nDriverRating : 1.0\nDeductible : 400.0\nPolicyType : 

In [31]:
pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 35.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 39.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]
Note: you may need to restart the kernel to use updated packages.


In [32]:
import os
from huggingface_hub import login

token = os.getenv("HF_TOKEN")
login(token=token)

In [ ]:
# Implementation du LLM pour test de reponse
# chargement du token
token = os.environ["HF_TOKEN"]

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

KeyError: 'HF_TOKEN'

In [ ]:
import os
print(os.environ.keys())

In [ ]:
completion = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
)

In [ ]:
# Reponse brute du LLM
reponse = completion.choices[0].message
print(reponse)

In [ ]:
# Reponse transformee
selected_indices_in_preselect = parse_llm_response(reponse.content, 10, 4)
print(selected_indices_in_preselect)

In [ ]:
data_for_rag = pd.concat([X_init, pd.DataFrame(y_init)], axis = 1)
data_for_rag.head(5)

In [ ]:



# ===========================
# BOUCLE D'ACTIVE LEARNING
# ===========================
results = []
iteration = 0

print("\n" + "="*60)
print("DÉBUT DE LA BOUCLE D'ACTIVE LEARNING")
print("="*60)

while len(X_init) < 5000 and len(X_pool) > 0:
    iteration += 1
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration} - Labels: {len(X_init)}")
    print(f"{'='*60}")
    
    # ========== RESET DES INDEX (CRUCIAL) ==========
    X_pool = X_pool.reset_index(drop=True)
    y_pool = y_pool.reset_index(drop=True)
    X_pool_US = X_pool_US.reset_index(drop=True)
    y_pool_US = y_pool_US.reset_index(drop=True)
    X_pool_LLM = X_pool_LLM.reset_index(drop=True)
    X_raw_pool_LLM = X_raw_pool_LLM.reset_index(drop=True)
    y_pool_LLM = y_pool_LLM.reset_index(drop=True)
    
    # ---- SMOTE ----
    X_init_smote, y_init_smote = smote.fit_resample(X_init, y_init)
    X_init_US_smote, y_init_US_smote = smote.fit_resample(X_init_US, y_init_US)
    X_init_LLM_smote, y_init_LLM_smote = smote.fit_resample(X_init_LLM, y_init_LLM)
    
    # ---- Entraînement ----
    model.fit(X_init_smote, y_init_smote)
    model_US.fit(X_init_US_smote, y_init_US_smote)
    model_LLM.fit(X_init_LLM_smote, y_init_LLM_smote)
    
    # ---- Évaluation ----
    y_test_pred = model.predict(X_test)
    y_test_proba = model.predict_proba(X_test)[:, 1]
    
    y_test_US_pred = model_US.predict(X_test)
    y_test_US_proba = model_US.predict_proba(X_test)[:, 1]
    
    y_test_LLM_pred = model_LLM.predict(X_test)
    y_test_LLM_proba = model_LLM.predict_proba(X_test)[:, 1]
    
    # Stockage
    results.append({
        "labels_used": len(X_init),
        "accuracy_random_sampling": accuracy_score(y_test, y_test_pred),
        "f1_random_sampling": f1_score(y_test, y_test_pred),
        "recall_random_sampling": recall_score(y_test, y_test_pred),
        "precision_random_sampling": precision_score(y_test, y_test_pred),
        "auc_random_sampling": roc_auc_score(y_test, y_test_proba),
        "accuracy_uncertainty_sampling": accuracy_score(y_test, y_test_US_pred),
        "f1_uncertainty_sampling": f1_score(y_test, y_test_US_pred),
        "recall_uncertainty_sampling": recall_score(y_test, y_test_US_pred),
        "precision_uncertainty_sampling": precision_score(y_test, y_test_US_pred),
        "auc_uncertainty_sampling": roc_auc_score(y_test, y_test_US_proba),
        "accuracy_llm_sampling": accuracy_score(y_test, y_test_LLM_pred),
        "f1_llm_sampling": f1_score(y_test, y_test_LLM_pred),
        "recall_llm_sampling": recall_score(y_test, y_test_LLM_pred),
        "precision_llm_sampling": precision_score(y_test, y_test_LLM_pred),
        "auc_llm_sampling": roc_auc_score(y_test, y_test_LLM_proba)
    })
    
    print(f"AUC - Random: {results[-1]['auc_random_sampling']:.4f} | "
          f"US: {results[-1]['auc_uncertainty_sampling']:.4f} | "
          f"LLM: {results[-1]['auc_llm_sampling']:.4f}")
    
    # ---- Prédictions sur pools ----
    y_pool_proba = model.predict_proba(X_pool)[:, 1]
    y_pool_US_proba = model_US.predict_proba(X_pool_US)[:, 1]
    y_pool_LLM_proba = model_LLM.predict_proba(X_pool_LLM)[:, 1]
    
    # ========== RANDOM SAMPLING ==========
    print("\n[Random Sampling]")
    if len(X_pool) >= 50:
        x_batch, X_pool_new, y_batch, y_pool_new = train_test_split(
            X_pool, y_pool, train_size=50, stratify=y_pool, random_state=42
        )
        X_init = pd.concat([X_init, x_batch], ignore_index=True)
        y_init = pd.concat([y_init, y_batch], ignore_index=True)
        X_pool = X_pool_new
        y_pool = y_pool_new
        print(f"  +50 échantillons")
    else:
        print(f"  Pool épuisé")
        break
    
    # ========== UNCERTAINTY SAMPLING ==========
    print("\n[Uncertainty Sampling]")
    if len(X_pool_US) > 0:
        y_init_US_proba = model_US.predict_proba(X_init_US)[:, 1]
        fpr, tpr, thresholds = roc_curve(y_init_US, y_init_US_proba)
        youden = tpr - fpr
        best_idx = np.argmax(youden)
        youden_threshold = thresholds[best_idx]
        
        dist_to_threshold = np.abs(y_pool_US_proba - youden_threshold)
        n_select = min(50, len(X_pool_US))
        query_idx = np.argsort(dist_to_threshold)[:n_select]
        
        X_new_US = X_pool_US.iloc[query_idx]
        y_new_US = y_pool_US.iloc[query_idx]
        X_init_US = pd.concat([X_init_US, X_new_US], ignore_index=True)
        y_init_US = pd.concat([y_init_US, y_new_US], ignore_index=True)
        
        # Suppression par masque
        mask = np.ones(len(X_pool_US), dtype=bool)
        mask[query_idx] = False
        X_pool_US = X_pool_US[mask].reset_index(drop=True)
        y_pool_US = y_pool_US[mask].reset_index(drop=True)
        
        print(f"  +{len(query_idx)} échantillons (seuil: {youden_threshold:.4f})")
    
    # ========== LLM SAMPLING ==========
    print("\n[LLM-based Sampling]")
    
    if len(X_pool_LLM) == 0:
        print("Pool épuisé")
        continue
    
    try:
        # Paramètres issus de PROMPT_CONFIG
        n_preselect = min(PROMPT_CONFIG["n_preselect"], len(X_pool_LLM))
        n_final_select = min(PROMPT_CONFIG["n_select"], n_preselect)
        
        # STEP 1 : Présélection par uncertainty sampling
        print(f"  Présélection: {n_preselect} candidats (US)")
        preselect_idx = np.random.choice(len(X_pool_LLM), size=n_preselect, replace=False)

        X_raw_preselect = X_raw_pool_LLM.iloc[preselect_idx].reset_index(drop=True)
        preselect_proba = y_pool_LLM_proba[preselect_idx]
        
        # STEP 2 : Sélection intelligente + RAG
        print(f"  Sélection intelligente de {n_final_select} parmi {n_preselect}...")
        data_for_rag = pd.concat([])
        text_samples = preprocessor_llm.transform(X_raw_preselect)
        prompt = create_llm_prompt(text_samples, n_select=n_final_select)

        completion = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
        )

        selected_indices_in_preselect = parse_llm_response(completion.choices[0].message.content, n_preselect)
        
        # Validation
        selected_indices_in_preselect = [
            int(idx) for idx in selected_indices_in_preselect
            if 0 <= int(idx) < n_preselect
        ]
        print(f"  ✓ {len(selected_indices_in_preselect)} indices valides")
        
        # STEP 3 : Conversion en indices pool
        final_query_idx = preselect_idx[selected_indices_in_preselect]
        
        # STEP 4 : Mise à jour
        X_new_LLM = X_pool_LLM.iloc[final_query_idx]
        y_new_LLM = y_pool_LLM.iloc[final_query_idx]
        X_init_LLM = pd.concat([X_init_LLM, X_new_LLM], ignore_index=True)
        y_init_LLM = pd.concat([y_init_LLM, y_new_LLM], ignore_index=True)
        
        mask = np.ones(len(X_pool_LLM), dtype=bool)
        mask[final_query_idx] = False
        X_pool_LLM = X_pool_LLM[mask].reset_index(drop=True)
        X_raw_pool_LLM = X_raw_pool_LLM[mask].reset_index(drop=True)
        y_pool_LLM = y_pool_LLM[mask].reset_index(drop=True)
        
        print(f"  +{len(final_query_idx)} échantillons | Pool restant: {len(X_pool_LLM)}")
        
    except Exception as e:
        print(f"  ❌ ERREUR LLM: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# ===========================
# RÉSULTATS
# ===========================
results_df = pd.DataFrame(results)

print("\n Résultats finaux:")
print(results_df.tail(10))

# Sauvegarder
results_df.to_csv('active_learning_results.csv', index=False)
print("\n Résultats sauvegardés: active_learning_results.csv")

# Afficher
display(results_df.tail())

In [ ]:
# ===========================
# VISUALISATION
# ===========================
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.0)
metrics = ["accuracy", "f1", "recall", "precision", "auc"]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    ax = axes[i]
    
    ax.plot(results_df["labels_used"], results_df[f"{metric}_random_sampling"], 
            'o-', label="Random", color=colors[0], linewidth=2, markersize=1)
    
    ax.plot(results_df["labels_used"], results_df[f"{metric}_uncertainty_sampling"], 
            's-', label="Uncertainty", color=colors[1], linewidth=2, markersize=1)
    
    ax.plot(results_df["labels_used"], results_df[f"{metric}_llm_sampling"], 
            '^-', label="LLM", color=colors[2], linewidth=2, markersize=1)
    
    ax.set_title(metric.upper(), fontsize=12, fontweight='bold')
    ax.set_xlabel("Labels utilisés")
    ax.set_ylabel(metric.upper())
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

fig.delaxes(axes[5])

plt.suptitle("Comparaison Active Learning: Random vs Uncertainty vs LLM", 
             fontsize=14, fontweight='bold')
plt.tight_layout()

plt.savefig('active_learning_results.png', dpi=150, bbox_inches='tight')
print("✓ Graphique sauvegardé: active_learning_results.png")
plt.show()